In [5]:
!uv add wfdb 

Resolved 105 packages in 0.59ms
Audited 102 packages in 0.59ms


In [6]:
import pandas as pd
df = pd.read_csv('../dataset/ptbxl_database.csv')

In [ ]:
print("✓ Colunas concatenadas para criar 'rotulo' com sucesso!")

KeyError: 'macro_class'

In [12]:
import pandas as pd
import numpy as np
import wfdb
import ast

def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

path = ''
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv('../dataset/ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))



# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv('../dataset/scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]
afib_repo = 'AFIB'
normal_repo = ['NORMAL','SR']
def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in afib_repo:
            tmp.append('AFIB')
        elif key in normal_repo:
            tmp.append('NORMAL')
        else:
            tmp.append('Other')
    return list(set(tmp))[-1]

# Apply diagnostic superclass
print(aggregate_diagnostic)
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)



<function aggregate_diagnostic at 0x7dfb295e00e0>


In [13]:
Y['diagnostic_superclass'].value_counts()

diagnostic_superclass
NORMAL    16748
Other      3542
AFIB       1509
Name: count, dtype: int64

In [16]:
## Hot encoding na coluna 'diagnostic_superclass'
Y = pd.get_dummies(Y, columns=['diagnostic_superclass'])
Y.head()

KeyError: "None of [Index(['diagnostic_superclass'], dtype='str')] are in the [columns]"

In [15]:

Y['path']=Y['filename_lr'].apply(lambda x: x.split('/')[-1]+'-0.png')

In [ ]:
## generate csv with columns path and diagnostic_superclass
Y[['path', 'diagnostic_superclass','patient_id']].to_csv('dataset_FA_Outros.csv', index=False)   

In [1]:
Y['path'].to_csv('dataset_FA_Outros.csv')

NameError: name 'Y' is not defined

In [19]:
Y.rename(columns={'diagnostic_superclass_AFIB': 'AFIB'}, inplace=True)
Y.rename(columns={'diagnostic_superclass_NORMAL': 'NORMAL'}, inplace=True)
Y.rename(columns={'diagnostic_superclass_Other': 'Other'}, inplace=True)

In [20]:
Y[['patient_id', 'path','age','sex','height','weight','AFIB', 'NORMAL', 'Other']].to_csv('dataset_FA_OUTROS_NORMAL_BINARY_HOT_ENCODING.csv', index=False)